In [0]:
import requests

# TVMaze Shows API
url = "https://api.tvmaze.com/shows"

response = requests.get(url, timeout=60)

print("HTTP Status:", response.status_code)

response.raise_for_status()

shows = response.json()

print("Total shows extracted:", len(shows))

In [0]:
STORAGE_ACCOUNT = "sttvmazede2026"

RAW_BASE = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net"

SHOWS_PATH = f"{RAW_BASE}/shows"
EPISODES_PATH = f"{RAW_BASE}/episodes"
CAST_PATH = f"{RAW_BASE}/cast"

print(RAW_BASE)

In [0]:
import json

shows_file = f"{SHOWS_PATH}/shows.json"

dbutils.fs.put(
    shows_file,
    json.dumps(shows, ensure_ascii=False),
    overwrite=True
)

print("Shows saved:")
print(shows_file)

In [0]:
import requests
import time

def get_api_data(url, retries=3):
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=60)

            if response.status_code == 200:
                return response.json()

            print("HTTP Status:", response.status_code)

        except Exception as e:
            print("Attempt failed:", e)

        time.sleep(2)

    return []

In [0]:
episodes = []

for index, show in enumerate(shows):

    show_id = show["id"]

    url = f"https://api.tvmaze.com/shows/{show_id}/episodes"

    data = get_api_data(url)

    for episode in data:
        episode["show_id"] = show_id
        episodes.append(episode)

    if (index + 1) % 25 == 0:
        print(
            f"Processed {index + 1} / {len(shows)} shows"
        )

print("Total episodes:", len(episodes))

In [0]:
episodes_file = f"{EPISODES_PATH}/episodes.json"

dbutils.fs.put(
    episodes_file,
    json.dumps(episodes, ensure_ascii=False),
    overwrite=True
)

print("Episodes saved:")
print(episodes_file)

In [0]:
cast_data = []

for index, show in enumerate(shows):

    show_id = show["id"]

    url = f"https://api.tvmaze.com/shows/{show_id}/cast"

    data = get_api_data(url)

    for item in data:

        person = item.get("person") or {}
        character = item.get("character") or {}

        cast_data.append({
            "show_id": show_id,
            "person_id": person.get("id"),
            "cast_name": person.get("name"),
            "character_id": character.get("id"),
            "character_name": character.get("name")
        })

    if (index + 1) % 25 == 0:
        print(
            f"Processed {index + 1} / {len(shows)} shows"
        )

print("Total cast records:", len(cast_data))

In [0]:
cast_file = f"{CAST_PATH}/cast.json"

dbutils.fs.put(
    cast_file,
    json.dumps(cast_data, ensure_ascii=False),
    overwrite=True
)

print("Cast saved:")
print(cast_file)

In [0]:
CATALOG = "tvmaze"

spark.sql(f"""
CREATE CATALOG IF NOT EXISTS {CATALOG}
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS tvmaze.bronze
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS tvmaze.silver
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS tvmaze.gold
""")

print("Catalog and schemas created")

In [0]:
shows_df = (
    spark.read
    .option("multiline", "true")
    .json(SHOWS_PATH + "/shows.json")
)

print("Shows:", shows_df.count())

display(shows_df.limit(10))

In [0]:
from pyspark.sql import functions as F

bronze_shows = (
    shows_df
    .withColumn("ingest_date", F.current_date())
    .withColumn("ingest_timestamp", F.current_timestamp())
)

bronze_shows.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tvmaze.bronze.bronze_shows")

In [0]:
episodes_df = (
    spark.read
    .option("multiline", "true")
    .json(EPISODES_PATH + "/episodes.json")
)

print("Episodes:", episodes_df.count())

display(episodes_df.limit(10))

In [0]:
bronze_episodes = (
    episodes_df
    .withColumn("ingest_date", F.current_date())
    .withColumn("ingest_timestamp", F.current_timestamp())
)

bronze_episodes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tvmaze.bronze.bronze_episodes")

In [0]:
cast_df = (
    spark.read
    .option("multiline", "true")
    .json(CAST_PATH + "/cast.json")
)

print("Cast:", cast_df.count())

display(cast_df.limit(10))

In [0]:
bronze_cast = (
    cast_df
    .withColumn("ingest_date", F.current_date())
    .withColumn("ingest_timestamp", F.current_timestamp())
)

bronze_cast.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tvmaze.bronze.bronze_cast")

In [0]:
from pyspark.sql import functions as F

shows = spark.table(
    "tvmaze.bronze.bronze_shows"
)

display(shows.limit(5))

In [0]:
silver_shows = (
    shows.select(
        F.col("id").cast("long").alias("show_id"),

        F.trim(
            F.col("name")
        ).alias("show_name"),

        F.trim(
            F.col("language")
        ).alias("language"),

        F.col("status").alias("status"),

        F.col("runtime")
        .cast("int")
        .alias("runtime"),

        F.to_date(
            F.col("premiered")
        ).alias("premiered"),

        F.col("genres").alias("genres"),

        F.col("rating.average")
        .cast("double")
        .alias("rating_average"),

        F.col("network.name")
        .alias("network_name")
    )
    .dropDuplicates(["show_id"])
)

In [0]:
silver_shows.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tvmaze.silver.silver_shows")

In [0]:
silver_show_genres = (
    silver_shows
    .select(
        "show_id",
        F.explode_outer("genres").alias("genre")
    )
    .filter(
        F.col("genre").isNotNull()
    )
    .withColumn(
        "genre",
        F.trim(F.col("genre"))
    )
    .dropDuplicates(
        ["show_id", "genre"]
    )
)

In [0]:
silver_show_genres.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "tvmaze.silver.silver_show_genres"
    )

In [0]:
episodes = spark.table(
    "tvmaze.bronze.bronze_episodes"
)

In [0]:
from pyspark.sql import functions as F

episodes = spark.table(
    "tvmaze.bronze.bronze_episodes"
)

silver_episodes = (
    episodes.select(

        F.col("id")
        .cast("long")
        .alias("episode_id"),

        F.col("show_id")
        .cast("long")
        .alias("show_id"),

        F.trim(
            F.col("name")
        ).alias("episode_name"),

        F.col("season")
        .cast("int")
        .alias("season"),

        F.col("number")
        .cast("int")
        .alias("episode_number"),

        # FIX: empty string becomes NULL before DATE conversion
        F.to_date(
            F.when(
                F.trim(F.col("airdate")) != "",
                F.trim(F.col("airdate"))
            )
        ).alias("airdate"),

        F.col("runtime")
        .cast("int")
        .alias("runtime"),

        F.col("rating.average")
        .cast("double")
        .alias("rating")
    )
)

In [0]:
silver_episodes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "tvmaze.silver.silver_episodes"
    )

In [0]:
shows = spark.table(
    "tvmaze.silver.silver_shows"
).alias("s")

episodes = spark.table(
    "tvmaze.silver.silver_episodes"
).alias("e")

genres = spark.table(
    "tvmaze.silver.silver_show_genres"
).alias("g")

cast = spark.table(
    "tvmaze.silver.silver_cast"
).alias("c")

In [0]:
fact_show_data = (
    episodes
    .join(
        shows,
        F.col("e.show_id") == F.col("s.show_id"),
        "inner"
    )
    .join(
        genres,
        F.col("e.show_id") == F.col("g.show_id"),
        "left"
    )
    .join(
        F.broadcast(cast),
        F.col("e.show_id") == F.col("c.show_id"),
        "left"
    )
    .select(
        F.col("e.show_id").alias("show_id"),
        F.col("s.show_name").alias("show_name"),
        F.col("s.language").alias("language"),
        F.col("g.genre").alias("genre"),
        F.col("e.season").alias("season"),
        F.col("e.episode_name").alias("episode_name"),
        F.col("e.airdate").alias("airdate"),
        F.col("e.runtime").alias("runtime"),
        F.col("c.cast_name").alias("cast_name"),
        F.col("c.character_name").alias("character_name")
    )
)

In [0]:
from pyspark.sql import functions as F

cast = spark.table(
    "tvmaze.bronze.bronze_cast"
)

silver_cast = cast.select(
    F.col("show_id").cast("long").alias("show_id"),
    F.col("person_id").cast("long").alias("person_id"),
    F.trim(F.col("cast_name")).alias("cast_name"),
    F.trim(F.col("character_name")).alias("character_name")
)

In [0]:
silver_cast.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "tvmaze.silver.silver_cast"
    )

In [0]:
%sql
select * from tvmaze.silver.fact_show_data

In [0]:
fact_show_data.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "tvmaze.silver.fact_show_data"
    )

In [0]:
fact_show_data.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("season") \
    .saveAsTable(
        "tvmaze.silver.fact_show_data"
    )

In [0]:
%sql
OPTIMIZE tvmaze.silver.fact_show_data;

In [0]:
%sql
OPTIMIZE tvmaze.silver.fact_show_data
ZORDER BY (show_id, airdate)

In [0]:
fact = spark.table(
    "tvmaze.silver.fact_show_data"
)

display(fact.limit(10))

In [0]:
episodes_per_season = (
    fact
    .groupBy(
        "show_id",
        "show_name",
        "season"
    )
    .agg(
        F.countDistinct(
            "episode_name"
        ).alias(
            "episodes_per_season"
        )
    )
)

In [0]:
episodes_per_season.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "tvmaze.gold.gold_episodes_per_season"
    )

In [0]:
avg_runtime = (
    fact
    .groupBy(
        "show_id",
        "show_name"
    )
    .agg(
        F.round(
            F.avg("runtime"),
            2
        ).alias("avg_runtime")
    )
)

In [0]:
avg_runtime.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "tvmaze.gold.gold_avg_runtime"
    )

In [0]:
top_cast = (
    fact
    .filter(F.col("cast_name").isNotNull())
    .groupBy("cast_name")
    .agg(
        F.countDistinct("show_id")
        .alias("show_count")
    )
)

In [0]:
from pyspark.sql.window import Window

cast_window = Window.orderBy(
    F.desc("show_count")
)

top_cast = top_cast.withColumn(
    "cast_rank",
    F.row_number().over(cast_window)
)

In [0]:
top_cast.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "tvmaze.gold.gold_top_cast"
    )

In [0]:
genre_counts = (
    fact
    .filter(F.col("genre").isNotNull())
    .groupBy("genre")
    .agg(
        F.countDistinct("show_id")
        .alias("show_count")
    )
)

In [0]:
genre_window = Window.orderBy(
    F.desc("show_count")
)

In [0]:
genre_counts = genre_counts.withColumn(
    "genre_rank",
    F.row_number().over(genre_window)
)

In [0]:
genre_counts.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "tvmaze.gold.gold_genres"
    )

In [0]:
%sql
SELECT
    show_id,
    show_name,
    season,
    COUNT(DISTINCT episode_name) AS episode_count,

    RANK() OVER (
        PARTITION BY show_id
        ORDER BY COUNT(DISTINCT episode_name) DESC
    ) AS season_rank

FROM tvmaze.silver.fact_show_data

GROUP BY
    show_id,
    show_name,
    season;

In [0]:
%sql
SELECT
    s.language,
    g.genre,
    COUNT(DISTINCT e.episode_id) AS episode_count,
    AVG(e.runtime) AS avg_runtime

FROM tvmaze.silver.silver_shows s

JOIN tvmaze.silver.silver_episodes e
    ON s.show_id = e.show_id

LEFT JOIN tvmaze.silver.silver_show_genres g
    ON s.show_id = g.show_id

GROUP BY
    s.language,
    g.genre

ORDER BY
    episode_count DESC;